# Lesson 01 - Johdatus tekoälyagentteihin

Tervetuloa ensimmäiseen oppituntiin kurssilla **AI Agents for Beginners**!

**Tekoälyagentti** on ohjelma, joka käyttää suurta kielimallia (LLM) päättelymoottorinaan ja voi suorittaa *toimia* todellisessa maailmassa — kutsumalla API-rajapintoja, kyselyillä tietokantoihin tai ajamalla koodia — saavuttaakseen tavoitteen käyttäjän puolesta.

Tässä muistikirjassa rakennat ensimmäisen agenttisi: **matka-agentin**, joka suosittelee lomakohteita. Matkan varrella opit miten:

1. Yhdistää Azure AI Foundry Agent Serviceen käyttämällä **Microsoft Agent Frameworkia**.
2. Antaa agentille **työkalu** — tavallisen Python-funktion, jota se voi kutsua.
3. Suorittaa agentti ja tarkastella sen vastausta.
4. Suoratoistaa agentin vastaus token kerrallaan.


## Asetus

Ennen tämän muistikirjan suorittamista varmista, että sinulla on:

1. **Azure AI Foundry -projekti**, jossa on käyttöön otettu chat-malli (esim. `gpt-4o-mini`).
2. **Kirjautunut sisään Azure CLI:llä** — suorita `az login` päätelaitteessasi.
3. **Asetetut vaaditut ympäristömuuttujat:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry -projektisi päätepiste.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — käyttöön otetun mallisi nimi.

Alla oleva solu asentaa tarvitsemasi Python-kirjastot.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ensimmäisen agenttisi luominen

Agentti tarvitsee kaksi asiaa:

- **Ohjeet**, jotka kertovat sille *kuka se on* ja *miten käyttäytyä* (järjestelmäkehote).
- **Työkalut** — Python-funktiot, jotka on koristeltu `@tool`-merkinnällä ja joita agentti voi kutsua saadakseen tietoa tai suorittaakseen toimintoja.

Alla määrittelemme yksinkertaisen työkalun, joka palauttaa listan suosituista lomakohteista. Agentti käyttää tätä työkalua, kun käyttäjä pyytää matkasuosituksia.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Toistuvat vastaukset

Interaktiivisemman kokemuksen saamiseksi voit **toistaa** agentin vastauksen. Täyden vastauksen odottamisen sijaan agentti lähettää tekstinpaloja sitä mukaa kun ne syntyvät. Tämä on erityisen hyödyllistä chat-käyttöliittymissä, joissa haluat näyttää tulosteen reaaliajassa.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Yhteenveto

Tässä oppitunnissa opit miten:

- **Luo tarjoaja**, joka yhdistää Azure AI Foundry Agent Serviceen `AzureAIProjectAgentProvider`-luokan avulla.
- **Määritä työkalu** `@tool`-koristelijan avulla, jotta agentti voi kutsua Python-funktioitasi.
- **Suorita agentti** käyttäjäviestillä ja tulosta sen vastaus.
- **Striimaa vastauksia** reaaliaikaista tulostusta varten.

Seuraavassa oppitunnissa tutkimme agenttipohjaisia kehyksiä syvemmin ja opimme, miten agentille annetaan tehokkaampia työkaluja ja monivaiheisen päättelyn kykyjä.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttäen tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Pyrimme tarkkuuteen, mutta otathan huomioon, että automaattikäännöksissä voi esiintyä virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä tulee pitää virallisena lähteenä. Tärkeiden tietojen osalta suositellaan ammattimaisen ihmiskääntäjän käyttöä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai virhetulkintojen seurauksista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
